# Organização do ambiente

Neste notebook vamos preparar o ambiente usado no projeto. Primeiro criamos o catálogo `v_credit`, os schemas `bronze`, `silver` e `gold` e o Volume onde os arquivos serão armazenados. No final, conferimos se todos os arquivos do case foram enviados.

In [0]:
catalog = "v_credit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"
landing_volume_name = "landing"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"
landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/{landing_volume_name}"

## Criação da infraestrutura

Usamos `IF NOT EXISTS` para que qualquer integrante do grupo consiga executar esta parte sem tentar criar novamente os objetos que já existem.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.{landing_volume_name}")

print(f"Catálogo: {catalog}")
print(f"Schemas: {bronze_schema}, {silver_schema}, {gold_schema}")
print(f"Landing Zone: {landing_path}")

Catálogo: v_credit
Schemas: v_credit.bronze, v_credit.silver, v_credit.gold
Landing Zone: /Volumes/v_credit/bronze/landing


## Validação dos arquivos de entrada

Antes de seguir para a Bronze, verificamos se os oito arquivos esperados estão no Volume. Se algum estiver faltando, o notebook mostra quais arquivos ainda precisam ser enviados.

In [0]:
expected_files = {
    "base_atendentes.csv", "base_motivos.csv", "canais.csv",
    "chamados.csv", "Chamados_Hora.CSV", "clientes.csv",
    "custos.csv", "pesquisa_satisfacao.csv",
}

existing_files = {file.name for file in dbutils.fs.ls(landing_path)}
missing_files = sorted(expected_files - existing_files)
unexpected_files = sorted(existing_files - expected_files)

assert not missing_files, (
    "Landing Zone incompleta. Faça upload de: " + ", ".join(missing_files)
)

print(f"[OK] {len(expected_files)} arquivos obrigatórios encontrados.")
if unexpected_files:
    print(f"[AVISO] Arquivos adicionais serão ignorados: {unexpected_files}")

[OK] 8 arquivos obrigatórios encontrados.
